In [4]:
!pip install matplotlib
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    accuracy_score,
    f1_score,
    matthews_corrcoef
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import math
import random


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [5]:
import pandas as pd

train = pd.read_csv("stratified_train.csv")
val = pd.read_csv("stratified_val.csv")
test = pd.read_csv("stratified_test.csv")

MIN_SAMPLES_PER_CLASS = 100

print(f"Filtering datasets to only include HLA classes with >= {MIN_SAMPLES_PER_CLASS} examples in the training set.")

hla_counts = train['HLA'].value_counts()
well_represented_hlas = hla_counts[hla_counts >= MIN_SAMPLES_PER_CLASS].index.tolist()

print(f"\nFound {len(well_represented_hlas)} well-represented HLA classes (out of {len(hla_counts)} total).")

original_train_size = len(train)
original_val_size = len(val)
original_test_size = len(test)

train = train[train['HLA'].isin(well_represented_hlas)]
val = val[val['HLA'].isin(well_represented_hlas)]
test = test[test['HLA'].isin(well_represented_hlas)]


print("\n--- Dataset Sizes After Filtering ---")
print(f"Original training set size: {original_train_size} -> New size: {len(train)}")
print(f"Original validation set size: {original_val_size} -> New size: {len(val)}")
print(f"Original test set size: {original_test_size} -> New size: {len(test)}")

print("\nThe 'train', 'val', and 'test' DataFrames have been updated.")

Filtering datasets to only include HLA classes with >= 100 examples in the training set.

Found 111 well-represented HLA classes (out of 135 total).

--- Dataset Sizes After Filtering ---
Original training set size: 327528 -> New size: 326672
Original validation set size: 18193 -> New size: 18149
Original test set size: 18193 -> New size: 18148

The 'train', 'val', and 'test' DataFrames have been updated.


In [6]:
import torch
print(torch.cuda.is_available())
from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM 


MODEL_NAME ="facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

vocabulary = tokenizer.get_vocab()
sorted_vocabulary = sorted(vocabulary.items(), key=lambda item: item[1])


def create_presentation_label(pic50_value):
    # An IC50 of 500 nM corresponds to a pIC50 of ~6.301
    return 1 if pic50_value > 6.301 else 0
train['Presentation'] = train['pIC50'].apply(create_presentation_label)
val['Presentation'] = val['pIC50'].apply(create_presentation_label)
test['Presentation'] = test['pIC50'].apply(create_presentation_label)


MAX_PEPTIDE_LEN = train['Epitope'].str.len().max() + 2
MAX_HLA_LEN = train['HLA_seq'].str.len().max() + 2
print(f"Max peptide length set to: {MAX_PEPTIDE_LEN}")
print(f"Max HLA length set to: {MAX_HLA_LEN}")

mhc_map = {1: 0, 2: 1} 


tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")

from torch.utils.data import Dataset
import random 
import ast
import torch.nn.functional as F


class EpitopeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_peptide_len, max_hla_len):
        self.df = dataframe.dropna(subset=['HLA_seq']).reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_peptide_len = max_peptide_len
        self.max_hla_len = max_hla_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        epitope_seq = list(row['Epitope'])
        original_tokens = epitope_seq[:]
        for i in range(len(epitope_seq)):
            if random.random() < 0.15: epitope_seq[i] = self.tokenizer.mask_token
        
        tokenized_input = self.tokenizer(" ".join(epitope_seq), max_length=self.max_peptide_len, padding='max_length', truncation=True, return_tensors='pt')
        tokenized_labels = self.tokenizer(" ".join(original_tokens), max_length=self.max_peptide_len, padding='max_length', truncation=True, return_tensors='pt')
        tokenized_hla = self.tokenizer(" ".join(list(row['HLA_seq'])), max_length=self.max_hla_len, padding='max_length', truncation=True, return_tensors='pt')
        
        return {
            'input_ids': tokenized_input['input_ids'].flatten(),
            'attention_mask': tokenized_input['attention_mask'].flatten(),
            'hla_ids': tokenized_hla['input_ids'].flatten(),
            'hla_mask': tokenized_hla['attention_mask'].flatten(),
            'labels': tokenized_labels['input_ids'].flatten(),
            'pic50_label': torch.tensor(row['pIC50'], dtype=torch.float),
            'binary_label': torch.tensor(row['Presentation'], dtype=torch.float)
        }

import torch.nn as nn
from transformers import EsmForMaskedLM


class CrossAttentionMLMModel(nn.Module):
    def __init__(self, model_name="facebook/esm2_t12_35M_UR50D", num_heads=16): 
        super().__init__()
        
        self.encoder = EsmModel.from_pretrained(model_name)
        
        
        self.mlm_head = EsmForMaskedLM.from_pretrained(model_name).lm_head

        hidden_size = self.encoder.config.hidden_size 
        
        
        self.cross_attention = nn.MultiheadAttention(embed_dim=hidden_size, num_heads=num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(0.1) 
        
        
        self.regression_head = nn.Sequential(
            nn.Linear(hidden_size, 256), 
            nn.GELU(),                   
            nn.Dropout(0.1),             
            nn.Linear(256, 1)            
        )

    def freeze_encoder(self):
        for param in self.encoder.parameters(): param.requires_grad = False
        print(">> ESM Encoder frozen.")

    def unfreeze_encoder(self):
        for param in self.encoder.parameters(): param.requires_grad = True
        print(">> ESM Encoder unfrozen.")

    def forward(self, input_ids, attention_mask, hla_ids, hla_mask):
        
        peptide_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        peptide_embeds = peptide_outputs.last_hidden_state
        
    
        hla_outputs = self.encoder(input_ids=hla_ids, attention_mask=hla_mask)
        hla_embeds = hla_outputs.last_hidden_state
        
       
        mlm_logits = self.mlm_head(peptide_embeds)
        
        
        interaction_output, _ = self.cross_attention(query=peptide_embeds, key=hla_embeds, value=hla_embeds)
        
        
        updated_peptide_embeds = self.layer_norm(peptide_embeds + self.dropout(interaction_output))
        

        cls_token = updated_peptide_embeds[:, 0, :]
        
        pic50_prediction = self.regression_head(cls_token)
        
        return mlm_logits, pic50_prediction


import torch
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

all_tokens = []
for epitope in train['Epitope']:
    tokens = tokenizer.encode(epitope, add_special_tokens=False)
    all_tokens.extend(tokens)

class_labels = np.unique(all_tokens)
weights = compute_class_weight(
    class_weight='balanced',
    classes=class_labels,
    y=all_tokens
)

class_weights = torch.ones(tokenizer.vocab_size)
for token_id, weight in zip(class_labels, weights):
    class_weights[token_id] = weight

print("Class weights calculated successfully.")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_squared_error, matthews_corrcoef
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import pearsonr, spearmanr


def calculate_ppv_top_n(y_true, y_score, top_n_percent=0.01):
    """Calculates the fraction of true positives in the top N% of predictions."""
    if len(y_true) == 0: return 0.0
    
  
    data = list(zip(y_true, y_score))
    data.sort(key=lambda x: x[1], reverse=True)
    

    cutoff = int(len(data) * top_n_percent)
    if cutoff == 0: cutoff = 1 
    

    top_data = data[:cutoff]
    
 
    true_positives = sum(1 for t, s in top_data if t == 1)
    return true_positives / len(top_data)


def plot_training_curves(history, stage_prefix, save_folder):
    
    metrics = ['loss', 'r2', 'rmse', 'auc', 'auprc', 'pcc', 'srcc', 'mlm_acc'] 
    metric_names = ['Loss', 'R-squared', 'RMSE', 'ROC-AUC', 'AUPRC', 'Pearson PCC', 'Spearman SRCC', 'MLM Accuracy']
    
    for i, metric in enumerate(metrics):
        train_data = history.get(f'train_{metric}', [])
        val_data = history.get(f'val_{metric}', [])
        
        if not train_data and not val_data: continue

        plt.figure(figsize=(10, 6))
        if train_data: plt.plot(train_data, 'o-', label='Training')
        if val_data: plt.plot(val_data, 'o-', label='Validation')
        
        plt.title(f'{stage_prefix}: {metric_names[i]}', fontsize=16)
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel(metric_names[i], fontsize=12)
        plt.legend()
        plt.grid(True)
        
        filename = f"{stage_prefix}_{metric}.png"
        plt.savefig(os.path.join(save_folder, filename))
        plt.close()

def train_model(model, train_loader, val_loader, epochs, lr, class_weights,
                mlm_weight=1.0, regression_weight=1.0, patience=10, save_path='best_model.pt'):
    
    weight_decay = 0.088 
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    early_stopper = EarlyStopping(patience=patience, save_path=save_path)

    loss_fn_mlm = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id, weight=class_weights.to(DEVICE))
    loss_fn_reg = nn.MSELoss()
    model.to(DEVICE)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_r2': [], 'val_r2': [],
        'train_rmse': [], 'val_rmse': [],
        'train_auc': [], 'val_auc': [],        
        'train_auprc': [], 'val_auprc': [],    
        'train_pcc': [], 'val_pcc': [],        
        'train_srcc': [], 'val_srcc': [],      
        'train_mlm_acc': [], 'val_mlm_acc': [],
        'train_mlm_f1': [], 'val_mlm_f1': [],
        'train_mlm_mcc': [], 'val_mlm_mcc': []
    }
    
    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")
  
        model.train(); total_train_loss = 0

        train_true_pic50, train_pred_pic50 = [], []
        train_true_bin, train_true_mlm, train_pred_mlm = [], [], []

        for batch in tqdm(train_loader, desc="Training"):
            optimizer.zero_grad()
            input_ids=batch['input_ids'].to(DEVICE); attention_mask=batch['attention_mask'].to(DEVICE)
            hla_ids=batch['hla_ids'].to(DEVICE); hla_mask=batch['hla_mask'].to(DEVICE)
            labels=batch['labels'].to(DEVICE); pic50_label=batch['pic50_label'].to(DEVICE)
            binary_label=batch['binary_label'].to(DEVICE) 

            mlm_logits, pic50_preds = model(input_ids, attention_mask, hla_ids, hla_mask)
            
            loss_mlm = loss_fn_mlm(mlm_logits.view(-1, tokenizer.vocab_size), labels.view(-1))
            loss_reg = loss_fn_reg(pic50_preds.squeeze(), pic50_label)
            
            combined_loss = (mlm_weight * loss_mlm) + (regression_weight * loss_reg)
            
            combined_loss.backward(); optimizer.step()
            total_train_loss += combined_loss.item()
    
            if regression_weight > 0:
                train_true_pic50.extend(pic50_label.cpu().numpy())
                train_pred_pic50.extend(pic50_preds.squeeze().detach().cpu().numpy())
                train_true_bin.extend(binary_label.cpu().numpy())

            if mlm_weight > 0:
                mask_indices = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
                if mask_indices[0].numel() > 0:
                    train_true_mlm.extend(labels[mask_indices].cpu().numpy())
                    train_pred_mlm.extend(torch.argmax(mlm_logits[mask_indices], dim=-1).detach().cpu().numpy())

        avg_train_loss = total_train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        print(f"Average Training Loss: {avg_train_loss:.4f}")

        if regression_weight > 0 and len(train_true_pic50) > 1:
            history['train_r2'].append(r2_score(train_true_pic50, train_pred_pic50))
            history['train_rmse'].append(np.sqrt(mean_squared_error(train_true_pic50, train_pred_pic50)))
  
            try:
                history['train_auc'].append(roc_auc_score(train_true_bin, train_pred_pic50))
                history['train_auprc'].append(average_precision_score(train_true_bin, train_pred_pic50))
                history['train_pcc'].append(pearsonr(train_true_pic50, train_pred_pic50)[0])
                history['train_srcc'].append(spearmanr(train_true_pic50, train_pred_pic50)[0])
            except: 
                history['train_auc'].append(0); history['train_auprc'].append(0)
                history['train_pcc'].append(0); history['train_srcc'].append(0)
            print(f"  Train Reg -> R2: {history['train_r2'][-1]:.3f} | RMSE: {history['train_rmse'][-1]:.3f} | AUC: {history['train_auc'][-1]:.3f}")
            print(f"             PCC: {history['train_pcc'][-1]:.3f} | SRCC: {history['train_srcc'][-1]:.3f}")
        if mlm_weight > 0 and train_true_mlm:
            acc = accuracy_score(train_true_mlm, train_pred_mlm)
            mcc = matthews_corrcoef(train_true_mlm, train_pred_mlm)
            f1 = f1_score(train_true_mlm, train_pred_mlm, average='weighted', labels=np.unique(train_pred_mlm), zero_division=0)
            history['train_mlm_acc'].append(acc); history['train_mlm_f1'].append(f1); history['train_mlm_mcc'].append(mcc)
            print(f"MLM -> Acc: {acc:.4f}, F1: {f1:.4f}, MCC: {mcc:.4f}")
        else:
            history['train_mlm_acc'].append(0); history['train_mlm_f1'].append(0); history['train_mlm_mcc'].append(0)

        model.eval(); total_val_loss = 0
        val_true_pic50, val_pred_pic50 = [], []
        val_true_bin, val_true_mlm, val_pred_mlm = [], [], []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validating"):
                input_ids=batch['input_ids'].to(DEVICE); attention_mask=batch['attention_mask'].to(DEVICE)
                hla_ids=batch['hla_ids'].to(DEVICE); hla_mask=batch['hla_mask'].to(DEVICE)
                labels=batch['labels'].to(DEVICE); pic50_label=batch['pic50_label'].to(DEVICE)
                binary_label=batch['binary_label'].to(DEVICE) 

                mlm_logits, pic50_preds = model(input_ids, attention_mask, hla_ids, hla_mask)
                
                loss_mlm = loss_fn_mlm(mlm_logits.view(-1, tokenizer.vocab_size), labels.view(-1))
                loss_reg = loss_fn_reg(pic50_preds.squeeze(), pic50_label)
                total_val_loss += ((mlm_weight * loss_mlm) + (regression_weight * loss_reg)).item()
                
                if regression_weight > 0:
                    val_true_pic50.extend(pic50_label.cpu().numpy())
                    val_pred_pic50.extend(pic50_preds.squeeze().cpu().numpy())
                    val_true_bin.extend(binary_label.cpu().numpy())

                if mlm_weight > 0:
                    mask_indices = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
                    if mask_indices[0].numel() > 0:
                        val_true_mlm.extend(labels[mask_indices].cpu().numpy())
                        val_pred_mlm.extend(torch.argmax(mlm_logits[mask_indices], dim=-1).cpu().numpy())

        avg_val_loss = total_val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        print(f"Average Validation Loss: {avg_val_loss:.4f}")

        if regression_weight > 0 and len(val_true_pic50) > 1:
            r2 = r2_score(val_true_pic50, val_pred_pic50)
            rmse = np.sqrt(mean_squared_error(val_true_pic50, val_pred_pic50))
            
            try:
                auc = roc_auc_score(val_true_bin, val_pred_pic50)
                auprc = average_precision_score(val_true_bin, val_pred_pic50)
                pcc = pearsonr(val_true_pic50, val_pred_pic50)[0]
                srcc = spearmanr(val_true_pic50, val_pred_pic50)[0]
                ppv1 = calculate_ppv_top_n(val_true_bin, val_pred_pic50, top_n_percent=0.01)
            except Exception as e:
                print(f"Metric Error: {e}"); auc=0; auprc=0; pcc=0; srcc=0; ppv1=0

            history['val_r2'].append(r2); history['val_rmse'].append(rmse)
            history['val_auc'].append(auc); history['val_auprc'].append(auprc)
            history['val_pcc'].append(pcc); history['val_srcc'].append(srcc)

            print(f"pIC50 -> R2: {r2:.3f} | RMSE: {rmse:.3f} | AUC: {auc:.3f} | AUPRC: {auprc:.3f}")
            print(f"         PCC: {pcc:.3f} | SRCC: {srcc:.3f} | PPV(1%): {ppv1:.3f}")
        else:
            history['val_r2'].append(0); history['val_rmse'].append(0)
            history['val_auc'].append(0); history['val_auprc'].append(0)
            history['val_pcc'].append(0); history['val_srcc'].append(0)
        
        if mlm_weight > 0 and val_true_mlm:
            acc = accuracy_score(val_true_mlm, val_pred_mlm)
            mcc = matthews_corrcoef(val_true_mlm, val_pred_mlm)
            f1 = f1_score(val_true_mlm, val_pred_mlm, average='weighted', labels=np.unique(val_pred_mlm), zero_division=0)
            history['val_mlm_acc'].append(acc); history['val_mlm_f1'].append(f1); history['val_mlm_mcc'].append(mcc)
            print(f"MLM   -> Acc: {acc:.3f}, F1: {f1:.3f}, MCC: {mcc:.3f}")
        else:
            history['val_mlm_acc'].append(0); history['val_mlm_f1'].append(0); history['val_mlm_mcc'].append(0)

        print("------------------------")
        early_stopper.check(avg_val_loss, model)
        if early_stopper.should_stop_now: print("Early stopping criterion met."); break
            
    print("\n--- Training Complete ---")
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    return model, history
class EarlyStopping:
    def __init__(self, patience, save_path):
        self.patience = patience
        self.save_path = save_path
        self.counter = 0
        self.best_validation_loss = np.inf
        self.should_stop_now = False

    def check(self, current_validation_loss, model):
        if current_validation_loss < self.best_validation_loss:
            self.best_validation_loss = current_validation_loss
            self.counter = 0
            print(f"New best model! Val loss: {current_validation_loss:.4f}. Saving to {self.save_path}")
            torch.save(model.state_dict(), self.save_path)
        else:
            self.counter += 1
            print(f"No improvement. EarlyStopping Counter: {self.counter} of {self.patience}")
            if self.counter >= self.patience:
                self.should_stop_now = True

from sklearn.metrics import classification_report

def run_final_evaluation(model, test_dataset, tokenizer, device):
    print("\n\n<<<<<<<<<< RUNNING FINAL EVALUATION ON TEST SET >>>>>>>>>>")
    test_loader = DataLoader(test_dataset, batch_size=32) 
    model.eval(); model.to(device)

    true_pic50, pred_pic50, true_bin = [], [], []
    true_aa, pred_aa = [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Final Evaluation..."):
            input_ids=batch['input_ids'].to(device); attention_mask=batch['attention_mask'].to(device)
            hla_ids=batch['hla_ids'].to(device); hla_mask=batch['hla_mask'].to(device)
            labels=batch['labels'].to(device); pic50_label=batch['pic50_label'].to(device)
            binary_label=batch['binary_label'].to(device) 

            mlm_logits, pic50_preds = model(input_ids, attention_mask, hla_ids, hla_mask)

            true_pic50.extend(pic50_label.cpu().numpy())
            pred_pic50.extend(pic50_preds.cpu().numpy().flatten())
            true_bin.extend(binary_label.cpu().numpy())

            mask_indices = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)
            if mask_indices[0].numel() > 0:
                true_aa.extend(labels[mask_indices].cpu().numpy())
                pred_aa.extend(torch.argmax(mlm_logits[mask_indices], dim=-1).cpu().numpy())

    print("\n--- Final Regression Performance ---")
    if true_pic50:
        rmse = np.sqrt(mean_squared_error(true_pic50, pred_pic50))
        r2 = r2_score(true_pic50, pred_pic50)
        pcc = pearsonr(true_pic50, pred_pic50)[0]
        srcc = spearmanr(true_pic50, pred_pic50)[0]
  
        auc = roc_auc_score(true_bin, pred_pic50)
        auprc = average_precision_score(true_bin, pred_pic50)
        ppv1 = calculate_ppv_top_n(true_bin, pred_pic50, top_n_percent=0.01)

        print(f"RMSE:    {rmse:.4f}")
        print(f"R2:      {r2:.4f}")
        print(f"PCC:     {pcc:.4f}")
        print(f"SRCC:    {srcc:.4f}")
        print(f"ROC-AUC: {auc:.4f}")
        print(f"AUPRC:   {auprc:.4f}")
        print(f"PPV(1%): {ppv1:.4f}")

    print("\n--- Final MLM Performance ---")
    if true_aa:
        acc = accuracy_score(true_aa, pred_aa)
        mcc = matthews_corrcoef(true_aa, pred_aa)
        print(f"MLM Accuracy: {acc:.4f}")
        print(f"MLM MCC:      {mcc:.4f}")
import sys
import os

class DualLogger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

if __name__ == '__main__':
    output_dir = "training_results_esm_small_model" 
    os.makedirs(output_dir, exist_ok=True)
    sys.stdout = DualLogger(os.path.join(output_dir, "training_log.txt"))
    print(f"Results and logs will be saved to: {output_dir}")
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {DEVICE}")
    
    train_dataset = EpitopeDataset(train, tokenizer, MAX_PEPTIDE_LEN, MAX_HLA_LEN)
    val_dataset = EpitopeDataset(val, tokenizer, MAX_PEPTIDE_LEN, MAX_HLA_LEN)
    
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256)

    model = CrossAttentionMLMModel(model_name="facebook/esm2_t6_8M_UR50D")

   
    
    
    

True


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Max peptide length set to: 39
Max HLA length set to: 36


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Class weights calculated successfully.
Results and logs will be saved to: training_results_esm_small_model
Using device: cuda


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.weight', 'esm.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Epoch 1/150 ---
Average Training Loss: 0.3539
MLM -> Acc: 0.1594, F1: 0.1677, MCC: 0.1190
Average Validation Loss: 0.3337
MLM   -> Acc: 0.218, F1: 0.228, MCC: 0.179
------------------------
New best model! Val loss: 0.3337. Saving to stage1_esm_small_model_.pt

--- Epoch 2/150 ---
Average Training Loss: 0.3122
MLM -> Acc: 0.2460, F1: 0.2571, MCC: 0.2119
Average Validation Loss: 0.2999
MLM   -> Acc: 0.280, F1: 0.288, MCC: 0.247
------------------------
New best model! Val loss: 0.2999. Saving to stage1_esm_small_model_.pt

--- Epoch 3/150 ---
Average Training Loss: 0.2831
MLM -> Acc: 0.3091, F1: 0.3179, MCC: 0.2772
Average Validation Loss: 0.2753
MLM   -> Acc: 0.333, F1: 0.342, MCC: 0.302
------------------------
New best model! Val loss: 0.2753. Saving to stage1_esm_small_model_.pt

--- Epoch 4/150 ---
Average Training Loss: 0.2609
MLM -> Acc: 0.3566, F1: 0.3630, MCC: 0.3262
Average Validation Loss: 0.2556
MLM   -> Acc: 0.377, F1: 0.385, MCC: 0.348
------------------------
New bes

In [7]:
model, history_s1 = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader,
        epochs=150, lr=1e-4, class_weights=class_weights,
        mlm_weight=1, regression_weight=0, 
        patience=10, save_path='stage1_esm_small_model_.pt'
    )
plot_training_curves(history_s1, "Stage 1", output_dir)

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

In [8]:
print("\n--- Stage 2: Prediction-focused Fine-tuning ---")
model, history_s2 = train_model(
    model=model, train_loader=train_loader, val_loader=val_loader,
    epochs=150, lr=5e-5, class_weights=class_weights,
    mlm_weight=1, regression_weight=0.25, 
    patience=10, save_path='stage2_esm_small_model_.pt'
)
plot_training_curves(history_s2, "Stage 2", output_dir)

print("\n--- Final Model Trained and Ready ---")

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]

Training:   0%|          | 0/1277 [00:00<?, ?it/s]

Validating:   0%|          | 0/71 [00:00<?, ?it/s]